In [ ]:
import os
import re
import pandas as pd
import numpy as np
import importlib

from kfirlib import pubmed
from kfirlib.comparisons import extract_letters

In [ ]:
importlib.reload(pubmed)

In [ ]:
DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Natalie\tables\\'

In [ ]:
ris_df = pd.read_excel(DIR + 'ris_with_decision.xlsx')
ris_df.head()

In [ ]:
by_pubmed_id = pubmed.add_lines_by_pubmed_id(ris_df[['pubmed_id']], 'pubmed_id')
by_pubmed_id.head()

In [ ]:
def first_non_na(s):
    s = s.dropna()
    return s.iloc[0] if len(s) else np.nan

ris_with_pubmed = ris_df[ris_df.pubmed_id.notna()]
ris_without_pubmed = ris_df[ris_df.pubmed_id.isna()]

ris_plus = pd.concat([ris_with_pubmed, by_pubmed_id], axis=0)
ris_plus = ris_plus.groupby('pubmed_id', as_index=False).agg(first_non_na)
ris_plus = pd.concat([ris_plus, ris_without_pubmed], axis=0)
len(ris_df), len(by_pubmed_id), len(ris_plus)

In [ ]:
ris_df.doi.isna().sum(), ris_plus.doi.isna().sum()

In [ ]:
pdfs_df = pd.read_excel(DIR + 'pdfs_decision.xlsx')

In [ ]:
ris_plus['title_extract'] = ris_plus['title'].apply(extract_letters)
pdfs_df['title_extract'] = pdfs_df['title'].apply(extract_letters)
ris_plus.columns

In [ ]:
pdfs_df.columns

In [ ]:
ris_plus.rename(columns={'reasoning': 'ris_reasoning', 'decision_title_abstract': 'ris_decision'}, inplace=True)
pdfs_df.rename(columns={'full_article_reasoning': 'pdf_reasoning', 'full_article_decision': 'pdf_decision'}, inplace=True)

In [ ]:
merged = pd.merge(ris_plus, pdfs_df, on='title_extract', how='outer')
merged.head()

In [ ]:
del merged['title_y']
merged.rename(columns={'title_x': 'title'}, inplace=True)
merged['has_both'] = merged['ris_decision'].notna() & merged['pdf_decision'].notna()
len(merged), merged['has_both'].sum()

In [ ]:
merged.to_excel(DIR + 'stupid_merge.xlsx', index=False)

In [ ]:
len(ris_df), len(pdfs_df)